In [0]:
# imports
from utils.common_utils import (
    read_file_as_df,
    write_df_to_delta_table,
    clean_column_names,
    check_data_quality,
)
from utils.transform_utils import (
    enrich_customers,
    enrich_products,
    enrich_orders,
    aggregate_orders,
)
from pyspark.sql import functions as F
import warnings

warnings.filterwarnings("ignore")  # ignoring warnings

In [0]:
# define data location, currently placed in managed volume
data_folder = "/Volumes/databricks_learning_01/default/data/"
orders_path = data_folder + "Order.json"
customers_path = data_folder + "Customer_Updated.xlsx"
products_path = data_folder + "Product.csv"

In [0]:
# Get default parallelism and adjust the shuffle partitions to optimize joins in the later stage
default_parallelism = spark.sparkContext.defaultParallelism
print(f"Default parallelism: {default_parallelism}")
spark.conf.set("spark.sql.shuffle.partitions", default_parallelism)

# enabling aqe for better performance
# note: aqe is supported post spark 3 release and code assumes its being ran on spark 3 and above cluster
spark.conf.set("spark.sql.adaptive.enabled", "true")

In [0]:
# read the required data and clean the column names and repartitioning orders to effectively utilize the cluster.
orders = read_file_as_df(
    file_format="json", location=orders_path, additional_options={"multiline": "true"}
)
orders = clean_column_names(orders)
orders = orders.repartition(default_parallelism)

products = read_file_as_df(
    file_format="csv",
    location=products_path,
    additional_options={"sep": ",", "header": "true", "quote": '"', "escape": '"'},
)
products = clean_column_names(products)
products.cache()

# cache customers to improve performance assuming customers dataset would be decently smaller
customers = read_file_as_df(file_format="excel", location=customers_path)
customers = clean_column_names(customers)

In [0]:
# create raw tables of the source data
write_df_to_delta_table(
    df=orders,
    catalog_name="databricks_learning_01",
    database_name="default",
    table_name="orders_raw",
    load_type="overwrite",
    schema_evolution_mode="overwriteSchema",
)

write_df_to_delta_table(
    df=products,
    catalog_name="databricks_learning_01",
    database_name="default",
    table_name="products_raw",
    load_type="overwrite",
    schema_evolution_mode="overwriteSchema",
)

write_df_to_delta_table(
    df=customers,
    catalog_name="databricks_learning_01",
    database_name="default",
    table_name="customers_raw",
    load_type="overwrite",
    schema_evolution_mode="overwriteSchema",
)

In [0]:
# enrich customers and check data quality
customers_enriched = customers.transform(enrich_customers)
customers_enriched = check_data_quality(
    df=customers_enriched,
    expected_condition=F.col("customer_name").isNull(),
    message="Column customer_name has null values",
)
write_df_to_delta_table(
    df=customers_enriched,
    catalog_name="databricks_learning_01",
    database_name="default",
    table_name="customers_enriched",
    load_type="overwrite",
    schema_evolution_mode="overwriteSchema",
)

In [0]:
# enrich products
products_enriched = products.transform(enrich_products)
products_enriched = check_data_quality(
    df=products_enriched,
    expected_condition=F.col("product_id").isNull(),
    message="Column product_id has null values",
)
write_df_to_delta_table(
    df=products_enriched,
    catalog_name="databricks_learning_01",
    database_name="default",
    table_name="products_enriched",
    load_type="overwrite",
    schema_evolution_mode="overwriteSchema",
)

In [0]:
# enrich orders
orders_enriched = enrich_orders(orders, products_enriched, customers_enriched)
orders_enriched = check_data_quality(
    df=orders_enriched,
    expected_condition=~F.col("order_date").rlike(r"\d{1,2}/\d{1,2}/\d{4}"),
    message="Column order_date not formatted correctly, expected format: dd/mm/yyyy",
)

# considering orders as fact table its better to partition on order date column since data might be huge.
write_df_to_delta_table(
    df=orders_enriched,
    catalog_name="databricks_learning_01",
    database_name="default",
    table_name="orders_enriched",
    load_type="overwrite",
    schema_evolution_mode="overwriteSchema",
    partition_columns=[
        "order_date"
    ],  # it makes more sense to keep it at yearmonth level rather than order_date
)

In [0]:
# aggregate orders to show profit by year,category,subcategory,customer
orders_aggregated = aggregate_orders(orders_enriched)

write_df_to_delta_table(
    df=orders_aggregated,
    catalog_name="databricks_learning_01",
    database_name="default",
    table_name="orders_aggregated",
    load_type="overwrite",
    schema_evolution_mode="overwriteSchema",
)

# create a view for running below sql queries as the final step
orders_aggregated.createOrReplaceTempView("orders_aggregated")

In [0]:
%sql
--Profit by Year
select
  year,
  sum(profit) as profit
from
  orders_aggregated
group by
  year
order by
  year;

In [0]:
%sql
--Profit by Year + Product Category
select
  year,
  category,
  sum(profit) as profit
from
  orders_aggregated
group by
  year,
  category
order by
  year,
  category;

In [0]:
%sql
--Profit by Customer
select
  customer_id,
  sum(profit) as profit
from
  orders_aggregated
group by
  customer_id
order by
  customer_id;

In [0]:
%sql
--Profit by Customer + Year
select
  customer_id,
  year,
  sum(profit) as profit
from
  orders_aggregated
group by
  customer_id,
  year
order by
  customer_id,
  year;

In [0]:
# clearing cached dataframes if any.
spark.catalog.clearCache()